# 05. Linear Attention, RWKV, RetNet, GLA, DeltaNet, and TTT

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Angelos Katharopoulos et al., 2020 (*Transformers are RNNs*); Yutao Sun et al., 2023 (*Retentive Network: RetNet*); Bo Peng et al., 2023 (*RWKV*); Imanol Schlag et al., 2021 (*DeltaNet*); Yu Sun et al., 2024 (*Test-Time Training: TTT*).
- **Purpose:** Swaps $(QK^\top)V \to Q(K^\top V)$ associativity to achieve $\mathcal{O}(T)$ linear time complexity and constant memory inference, augmented with explicit multi-scale decay (RetNet) and online delta error correction (DeltaNet).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Mamba Selective SSM and Mamba2 SSD](04_mamba_selective_ssm_and_mamba2_ssd.ipynb) $\to$ `🔗 Downstream Syntheses:` [Memory Architectures NTM PKM PEER Infini and Titans](06_memory_architectures_ntm_pkm_peer_infini_and_titans.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Multi-Hop Pointer Dereferencing:** Full quadratic softmax attention retains sharper recall over arbitrarily positioned historical tokens compared to bounded-capacity linear associative states.
> - **Bidirectional Masked Language Modeling:** Optimized for causal left-to-right generation; bidirectional tasks benefit from unmasked attention.

## Mathematical Foundations & Governing Equations

### 1. Retention Networks (RetNet Dual Modes)
- **Recurrent Mode ($\mathcal{O}(1)$ step memory):**
$$
S_t = \gamma S_{t-1} + k_t^\top v_t \in \mathbb{R}^{D_k \times D_v}, \quad o_t = q_t S_t \in \mathbb{R}^{D_v}
$$
- **Parallel Mode ($\mathcal{O}(T)$ training):**
$$
O = ((Q K^\top) \odot D) V, \quad D_{i, j} = \gamma^{i - j} \quad (i \ge j)
$$

### 2. DeltaNet Online Associative Update
Rather than pure additive memory accumulation, DeltaNet updates its memory matrix via online gradient descent on key-value prediction error:
$$
S_t = S_{t-1} + \beta_t (v_t - S_{t-1} k_t) k_t^\top
$$

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

q, k, v = weights["q"], weights["k"], weights["v"]
gamma = float(weights["gamma"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, Q shape: {list(q.shape)}, gamma={gamma}")

In [ ]:
def retnet_recurrent_forward(q, k, v, gamma):
    B, T, D_k = q.shape
    D_v = v.shape[-1]
    s_t = torch.zeros(B, D_k, D_v, device=q.device, dtype=q.dtype)
    outs = []
    for t in range(T):
        qt = q[:, t, :].unsqueeze(1)
        kt = k[:, t, :].unsqueeze(1)
        vt = v[:, t, :].unsqueeze(1)
        s_t = gamma * s_t + torch.bmm(kt.transpose(1, 2), vt)
        ot = torch.bmm(qt, s_t).squeeze(1)
        outs.append(ot.unsqueeze(1))
    return torch.cat(outs, dim=1), s_t

def retnet_parallel_forward(q, k, v, gamma):
    scores = torch.bmm(q, k.transpose(1, 2))
    indices = torch.arange(q.shape[1], device=q.device)
    diff = indices.unsqueeze(1) - indices.unsqueeze(0)
    decay = torch.where(diff >= 0, gamma ** diff.float(), torch.zeros_like(diff, dtype=torch.float32))
    return torch.bmm(scores * decay.unsqueeze(0), v)

def deltanet_forward(q, k, v, beta):
    B, T, D_k = q.shape
    D_v = v.shape[-1]
    s_t = torch.zeros(B, D_k, D_v, device=q.device, dtype=q.dtype)
    outs = []
    for t in range(T):
        qt = q[:, t, :].unsqueeze(1)
        kt = k[:, t, :].unsqueeze(2)
        vt = v[:, t, :].unsqueeze(1)
        b_val = beta[:, t].unsqueeze(1).unsqueeze(2)
        pred_v = torch.bmm(kt.transpose(1, 2), s_t)
        err_v = vt - pred_v
        s_t = s_t + b_val * torch.bmm(kt, err_v)
        ot = torch.bmm(qt, s_t).squeeze(1)
        outs.append(ot.unsqueeze(1))
    return torch.cat(outs, dim=1), s_t

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
ret_rec_out, ret_s = retnet_recurrent_forward(q, k, v, gamma)
ret_par_out = retnet_parallel_forward(q, k, v, gamma)
err_rec = float(torch.max(torch.abs(ret_rec_out - weights["ret_recurrent_out"])))
err_par = float(torch.max(torch.abs(ret_par_out - weights["ret_parallel_out"])))
err_s = float(torch.max(torch.abs(ret_s - weights["ret_final_s"])))
assert err_rec < 1e-6, f"RetNet recurrent error: {err_rec:.2e}"
assert err_par < 1e-6, f"RetNet parallel error: {err_par:.2e}"
assert err_s < 1e-6, f"RetNet final state error: {err_s:.2e}"

delta_out, delta_s = deltanet_forward(q, k, v, weights["beta"])
err_del = float(torch.max(torch.abs(delta_out - weights["delta_out"])))
assert err_del < 1e-6, f"DeltaNet error: {err_del:.2e}"
print(f"Parity Verified: RetNet Recurrent={err_rec:.2e}, Parallel={err_par:.2e}, DeltaNet={err_del:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_rec = benchmark_fn(lambda: retnet_recurrent_forward(q, k, v, gamma), iters=50, name="RetNet Recurrent")
b_par = benchmark_fn(lambda: retnet_parallel_forward(q, k, v, gamma), iters=50, name="RetNet Parallel")
b_del = benchmark_fn(lambda: deltanet_forward(q, k, v, weights["beta"]), iters=50, name="DeltaNet Update")
print(format_benchmark_table([b_rec, b_par, b_del]))

# Visualization: Memory State Heatmap & Causal Retention Decay
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].imshow(ret_s[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[0].set_title("RetNet State Matrix S [D_k, D_v]")
ax[1].imshow(delta_s[0].detach().numpy(), cmap="plasma", aspect="auto")
ax[1].set_title("DeltaNet State Matrix S [D_k, D_v]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unbounded Decay Factor (gamma >= 1.0)
# Over long sequences, gamma >= 1 causes memory accumulation to overflow without bounded saturation.
gamma_unstable = 1.05
s_unstable = torch.tensor([1.0])
for _ in range(25):
    s_unstable = gamma_unstable * s_unstable + 0.1

print(f"Unstable state value after 25 steps: {s_unstable.item():.2e}")
assert s_unstable.item() > 5.0, "Decay factor gamma > 1 must cause unbounded state expansion!"
print("Demonstrated requirement of strictly contractive decay gamma < 1.0 in Retention Networks.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Associative Linear Attention:** Rearranging $Q(K^\top V)$ achieves $\mathcal{O}(T)$ linear time and constant memory decoding.
2. **Multi-Scale Retention:** RetNet's exponential decay $\gamma \in (0, 1)$ stabilizes long-range context without catastrophic forgetting.
3. **Delta Error Correction:** DeltaNet and TTT update associative states via online gradient descent, dynamically purging obsolete keys.

---
### Cross-Domain Generalisation Matrix
- **Continual Online Learning:** Online delta updates dynamically adapt model weights to non-stationary data streams.
- **Real-Time Edge Audio:** Constant memory footprint allows running autoregressive speech synthesis on microcontrollers.
- **Robotics Trajectory Tracking:** Dynamic decay rates naturally down-weight stale sensor telemetry.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.md)
- **PyTorch 2.x Module:** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.py)
- **Functional JAX Module:** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.py)
- **Candle Rust Module:** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.rs)
- **Hardware Kernel:** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.rs)
- **Universal Golden SafeTensors:** [05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 04. Mamba Selective SSM and Mamba2 SSD](04_mamba_selective_ssm_and_mamba2_ssd.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 06. Memory Architectures NTM PKM PEER Infini and Titans](06_memory_architectures_ntm_pkm_peer_infini_and_titans.ipynb) &rarr;